# Introduzione al notebook
In questo notebook sperimentiamo la **quantizzazione** di modelli LLM e la loro esecuzione locale su CPU usando **Ollama** e modelli in formato **GGUF**.

Concetti teorici:
- **Quantizzazione**: riduzione della precisione numerica (es. da 16 bit a 8 o 4 bit) per diminuire memoria e tempo di calcolo.
- **Inferenza locale**: il modello gira sul PC dello studente, senza inviare dati al cloud, garantendo privacy.
- **Formato GGUF**: struttura ottimizzata per modelli compatibili con `llama.cpp` e Ollama.

Obiettivo:
- Scaricare modelli già quantizzati
- Eseguire inferenze
- Misurare prestazioni e qualità
- Riflettere sul compromesso tra accuratezza e velocità

## Installazione e verifica di Ollama
Concetto teorico:
- Ollama è un runtime per modelli LLM ottimizzato per CPU, compatibile con GGUF.
- Funziona anche su PC senza GPU, sfruttando la libreria `llama.cpp`.

Cosa fa il codice:
- Verifica che Ollama sia installato e funzionante, mostrando la versione.
- Se non installato, lo studente deve scaricarlo dal sito ufficiale.

Cosa si può capire:
- Conferma dell'ambiente di lavoro.
- Assicurarsi che il runtime sia pronto per scaricare e usare i modelli.

In [1]:
!pip install ollama


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!ollama --version

ollama version is 0.35.0


## Download di modelli quantizzati in GGUF
Concetto teorico:
- I modelli LLM possono essere distribuiti in vari formati e livelli di quantizzazione.
- La quantizzazione riduce dimensioni e velocizza l’inferenza, ma può degradare leggermente la qualità.

Cosa fa il codice:
- Scarica due versioni dello stesso modello LLaMA 2 da 3B parametri:
  - `q8_0` (8 bit)
  - `q4_0` (4 bit)
- Queste versioni serviranno per confrontare prestazioni e qualità.

Cosa si può capire:
- Differenze di dimensione e velocità tra INT8 e INT4.
- Effetto della quantizzazione sul tempo di download e spazio disco.

In [3]:
# Scarica i modelli
!ollama pull llama3.2:3b-instruct-q8_0
!ollama pull llama3.2:3b-instruct-q4_0
!ollama pull llama3.2:3b-instruct-q2_K

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling ed5cd7dbde6e: 100% ▕██████████████████▏ 3.4 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B

In [4]:
!ollama pull gpt-oss:20b #serve avere un modello più grande per fare il fine-tuning del modello più piccolo, quindi scarichiamo anche questo modello più grande

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling e7b273f96360: 100% ▕██████████████████▏  13 GB                         
pulling fa6710a93d78: 100% ▕██████████████████▏ 7.2 KB                         
pulling f60356777647: 100% ▕██████████████████▏  11 KB                         
pulling d8ba2f9a17b3: 100% ▕██████████████████▏   18 B                         
pulling 776beb3adb23: 100% ▕██████████████████▏  489 B                         
verifying sha256 digest 
writing manifest 
success 


In [5]:
# Lista i modelli scaricati
!ollama list 

NAME                         ID              SIZE      MODIFIED           
gpt-oss:20b                  17052f91a42e    13 GB     49 seconds ago        
llama3.2:3b-instruct-q2_K    43e1dcc10fd4    1.4 GB    57 seconds ago        
llama3.2:3b-instruct-q4_0    9b9453afbdd6    1.9 GB    58 seconds ago        
llama3.2:3b-instruct-q8_0    e410b836fe61    3.4 GB    About a minute ago    
qwen2.5:latest               845dbda0ea48    4.7 GB    7 hours ago           
llama3.2:latest              a80c4f17acd5    2.0 GB    33 hours ago          
nomic-embed-text:latest      0a109f422b47    274 MB    34 hours ago          
qwen3.5:2b                   324d162be6ca    2.7 GB    5 months ago          


## Prima inferenza di prova
Concetto teorico:
- L'inferenza è il processo di generazione di testo a partire da un prompt.
- Su CPU, la velocità dipende da quantizzazione, dimensione modello e ottimizzazione.

Cosa fa il codice:
- Definisce una funzione per eseguire un prompt con Ollama.
- Misura tempo di esecuzione e memoria RAM usata.
- Esegue un prompt semplice (riassunto sulla fotosintesi).

Cosa si può capire:
- Quanto velocemente un modello quantizzato risponde.
- Quanta RAM è richiesta in un caso reale.
- Relazione tra prompt, tempo di risposta e consumo di memoria.

In [6]:
import ollama
import time
import psutil
#in questa funzione eseguiamo il prompt con Ollama e misuriamo il tempo di esecuzione e la memoria utilizzata
def run_ollama_prompt(model_name, prompt):
    start_time = time.time()
    process = psutil.Process()  # processo corrente (Python)
    mem_used = 0

    # Esegui prompt con Ollama (API Python)
    response = ""
    response = ollama.chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}]
    )

    mem_used = max(mem_used, process.memory_info().rss / (1024**2))

    elapsed = time.time() - start_time
    return response["message"]["content"].strip(), elapsed, mem_used

model2 = "llama3.2:3b-instruct-q2_K"
model8 = "llama3.2:3b-instruct-q8_0"
model4 = "llama3.2:3b-instruct-q4_0"

models = [model2, model8, model4]

for model in models:
    output, elapsed, mem_used = run_ollama_prompt(
        model,
        "Raccontami 3 barzellette."
    )

    print(f"\n=== RISPOSTA {model } ===")
    print(output)
    print(f"\nTempo totale: {elapsed:.2f} s")
    print(f"Memoria massima: {mem_used:.2f} MB")


=== RISPOSTA llama3.2:3b-instruct-q2_K ===
Certo, qui sono alcune barzellette che potrebbero farti sorridere:

1.  A un amico che stava cercendo di spaventare il suo figlio, ma per caso, il bambino non era affezionato, ma a casa si era sconfessionato e stava iniziando a perdere per via della sua affezionevolezza.  L'ho di è in una parola (di qui) è la parola, cioè il suo è il suo e io sono l'io, ma io che ho lui, cioè io che ho lui, ma lui che glielo dà, cioè lui che glielo dà.  L'è questo (io so) che stavo di qua a chiedere se potevo avere la mia affezione, cioè se potevo avere la mia affezione.  Ma non, non è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì, sì, sì, è il mio.  Sì,

## Confronto tra modelli INT8 e INT4
Concetto teorico:
- I modelli quantizzati a 8 bit mantengono più precisione, ma sono più lenti e pesanti.
- I modelli a 4 bit sono più leggeri e veloci, ma possono perdere qualità.

Cosa fa il codice:
- Esegue una serie di prompt su entrambi i modelli.
- Registra tempo, memoria e risposta.
- Organizza i risultati in una tabella.

Cosa si può capire:
- Differenze di latenza e RAM tra 8 bit e 4 bit.
- Variazioni qualitative nelle risposte.
- Osservare il compromesso tra prestazioni e accuratezza.

In [7]:
prompts = [
    "Spiega cos'è la quantizzazione nei modelli di intelligenza artificiale.",
    "Traduci in inglese: La scuola è finita.",
    "Quali sono i principali vantaggi dell'inferenza locale?"
]

results = []

for precision_model in [model8, model4]:
    for p in prompts:
        out, t, mem = run_ollama_prompt(precision_model, p)
        results.append({
            "modello": precision_model,
            "prompt": p,
            "tempo_s": t,
            "memoria_MB": mem,
            "risposta": out
        })

import pandas as pd
df = pd.DataFrame(results)
df


,modello,prompt,tempo_s,memoria_MB,risposta
0,llama3.2:3b-instruct-q8_0,Spiega cos'è la quantizzazione nei modelli di ...,33.380215,73.062500,La quantizzazione è un concetto fondamentale n...
1,llama3.2:3b-instruct-q8_0,Traduci in inglese: La scuola è finita.,1.357034,73.070312,"Here's the translation:\n\n""The school is over..."
2,llama3.2:3b-instruct-q8_0,Quali sono i principali vantaggi dell'inferenz...,23.489359,73.082031,L'inferenza locale è un concetto matematico ch...
3,llama3.2:3b-instruct-q4_0,Spiega cos'è la quantizzazione nei modelli di ...,20.192648,73.097656,La quantizzazione è un concetto fondamentale n...
4,llama3.2:3b-instruct-q4_0,Traduci in inglese: La scuola è finita.,1.139154,73.105469,"The translation of ""La scuola è finita"" in Eng..."
5,llama3.2:3b-instruct-q4_0,Quali sono i principali vantaggi dell'inferenz...,13.055426,73.109375,L'inferenza locale è un concetto fondamentale ...


# Laboratorio: Quantizzazione dei modelli LLM (4-bit vs 8-bit)

## 🧩 Esercizio 2 – Misurare la velocità di inferenza

**Obiettivo:** comprendere come la quantizzazione influisce sulla **velocità di risposta** del modello.

**Istruzioni:**
1. Usa lo **stesso prompt breve**, ad esempio:
   > "Chi ha scritto *I Promessi Sposi*?"
2. Misura il **tempo totale di generazione della risposta** per ciascuna variante: **modello 8-bit** e **modello 4-bit**.
3. Ripeti la misura **3 volte** per ogni modello e calcola la **media**.

💡 *Suggerimento tecnico:* puoi misurare il tempo con:
```python
import time

prompt = "Chi ha scritto I Promessi Sposi?"

# Misura per il modello 8-bit
t8 = []
for _ in range(3):
    start = time.time()
    _ = model_8bit(prompt)
    t8.append(time.time() - start)
print(f"Tempo medio 8-bit: {sum(t8)/3:.2f} s")

# Misura per il modello 4-bit
t4 = []
for _ in range(3):
    start = time.time()
    _ = model_4bit(prompt)
    t4.append(time.time() - start)
print(f"Tempo medio 4-bit: {sum(t4)/3:.2f} s")
```

📘 *Domande guida:*
- Quale versione è più veloce in media?
- Il guadagno di velocità corrisponde a  “riduzione dei tempi di inferenza”?


---

## 🧩 Esercizio 3 – Valutazione qualitativa della risposta (accuratezza)

**Obiettivo:** osservare l’effetto della quantizzazione sulla **qualità linguistica** delle risposte.

**Istruzioni:**
Prepara **5 prompt**, uno per ciascun tipo:
1. Domanda fattuale breve  
2. Spiegazione di un concetto semplice  
3. Riassunto di un testo di 6–8 frasi  
4. Produzione creativa (es. inizio di un racconto)  
5. Ragionamento leggero (es. “Se Marco ha 3 mele e ne dà una a Luca…”)

Esegui i prompt su **modello 8-bit** e **modello 4-bit**, confrontando:
- coerenza del testo,  
- completezza delle risposte,  
- correttezza delle informazioni.

💡 *Suggerimento tecnico:* salva le risposte in un dizionario per confrontarle facilmente:
```python
prompts = [
    "Chi ha inventato il telefono?",
    "Spiega in modo semplice cos'è una rete neurale.",
    "Riassumi questo testo: [incolla testo di 6-8 frasi]",
    "Scrivi l'inizio di una storia ambientata su Marte.",
    "Se Marco ha 3 mele e ne dà una a Luca, quante ne rimangono?"
]

risposte = {"8bit": [], "4bit": []}
for p in prompts:
    risposte["8bit"].append(model_8bit(p))
    risposte["4bit"].append(model_4bit(p))
```

📘 *Domande guida:*
- Quale versione produce risposte più coerenti e accurate?
- Si nota perdita di informazione o ripetizioni nella versione 4-bit?
- Come si manifesta il “trade-off” tra efficienza e qualità?

---

## 🧩 Esercizio 4 – Prompt di stress (robustezza con input lunghi)

**Obiettivo:** valutare come la **quantizzazione più spinta (4-bit)** reagisce a input complessi.

**Istruzioni:**
1. Prendi un testo di circa **200–300 parole** (es. un breve articolo o un estratto da Wikipedia).
2. Crea un prompt che chieda di **riassumere** o **estrarre i punti chiave**.
3. Esegui lo stesso prompt con **modello 8-bit** e **modello 4-bit**.

💡 *Suggerimento tecnico:*
```python
long_prompt = "Riassumi il seguente testo in 5 punti chiave:\n\n[TUO TESTO QUI]"
print("Risposta 8-bit:")
print(model_8bit(long_prompt))
print("\nRisposta 4-bit:")
print(model_4bit(long_prompt))
```

📘 *Domande guida:*
- Il modello 4-bit mostra “allucinazioni” (informazioni inventate) o omissioni?
- La coerenza e la qualità del riassunto peggiorano rispetto all’8-bit?

---

## 🧩 Esercizio 5 – Misurare l’utilizzo di memoria in pratica

**Obiettivo:** verificare la riduzione del consumo RAM dovuta alla quantizzazione.

**Istruzioni:**
1. Avvia una sessione di inferenza con il **modello 8-bit** e poi con il **modello 4-bit**.
2. Durante l’esecuzione, osserva l’uso di memoria tramite strumenti di sistema (es. `top`, `htop` o Monitoraggio Attività su macOS).
3. Annota il valore massimo di RAM utilizzata e calcola la **differenza percentuale**.

💡 *Suggerimento tecnico:*
```python
import subprocess

def get_model_size_from_ps(model_name):
    """
    Legge l'output di 'ollama ps' e restituisce la memoria
    occupata dal modello (es. '2.8 GB').
    """

    result = subprocess.run(
        ["ollama", "ps"],
        capture_output=True,
        text=True
    )

    output = result.stdout

    for line in output.splitlines():

        if model_name in line:

            match = re.search(r"(\d+(\.\d+)?)\s*(GB|MB)", line)

            if match:
                return f"{match.group(1)} {match.group(3)}"

            return "Dimensione non trovata"

    return "Modello non trovato"

def unload_model(model_name):

    subprocess.run(
        ["ollama", "stop", model_name],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )


run_prompt(model, PROMPT)

time.sleep(2)

size = get_model_size_from_ps(model)

print(f"Memoria occupata: {size}")

results[model] = size

unload_model(model)  # ollama rimane in memoria per un po' il modello dopo averlo usato

time.sleep(5)

print(f"Memoria occupata: {size}")

```

📘 *Domande guida:*
- Quanta memoria risparmia il modello 4-bit?

---

## 🧩 Esercizio 6 – Stabilità su sessioni prolungate

**Obiettivo:** valutare la **coerenza nel tempo** dei modelli quantizzati.

**Istruzioni:**
1. Scegli un argomento (es. “Rivoluzione industriale” o “Evoluzione delle stelle”).
2. Simula una **conversazione di 10–15 turni** con entrambi i modelli.
3. Analizza se il modello 4-bit perde coerenza o produce risposte più vaghe col passare dei turni.

💡 *Suggerimento tecnico:* puoi simulare una conversazione concatenando le risposte:
```python
DOMANDE = [
    "Parliamo dell'evoluzione delle stelle. Dammi una panoramica generale.",
    "Come nasce una stella?",
    "Cos'è una nebulosa?",
    "Qual è il ruolo della gravità?",
    "Che differenza c'è tra una protostella e una stella adulta?",
    "Come funziona la fusione nucleare?"
    "Come si forma un buco nero stellare?",
    "Riassumi tutto quello che abbiamo discusso finora.",
    "Puoi costruire una timeline delle fasi evolutive?",
]

def run_conversation(model_name):

    messages = []

    transcript = []

    for turn, question in enumerate(DOMANDE, start=1):

        messages.append(
            {
                "role": "user",
                "content": question
            }
        )

        response = ollama.chat(
            model=model_name,
            messages=messages
        )

        answer = response["message"]["content"]

        messages.append(
            {
                "role": "assistant",
                "content": answer
            }
        )

        transcript.append({
            "turn": turn,
            "question": question,
            "answer": answer
        })

        print(f"\n=== TURNO {turn} ===")
        # print(f"DOMANDA: {question}")
        # print(f"RISPOSTA:\n{answer}")

    return transcript
```

📘 *Domande guida:*
- Il modello 4-bit rimane coerente sul tema?
- Ripete concetti o cambia argomento più spesso rispetto all’8-bit?
- Collega le osservazioni alle *best practices* di test dei modelli quantizzati.

---

## 🧩 Esercizio 7 – Valutazione della latenza percepita

**Obiettivo:** misurare la **reattività percepita** dal punto di vista dell’utente.

**Istruzioni:**
1. Cronometra:
   - il tempo tra l’invio del prompt e la comparsa del **primo token**,
   - il tempo totale per una risposta di circa **150 parole**.
2. Ripeti per **modello 8-bit** e **modello 4-bit**.
3. Valuta la “usabilità” con la scala:
   - 🔵 Ottimo: avvio < 1 s  
   - 🟢 Accettabile: 1–3 s  
   - 🔴 Scarso: > 3 s

💡 *Suggerimento tecnico:*
```python
start = time.time()
_ = model_8bit("Spiega in 150 parole come funziona la quantizzazione nei LLM.")
t_8 = time.time() - start

start = time.time()
_ = model_4bit("Spiega in 150 parole come funziona la quantizzazione nei LLM.")
t_4 = time.time() - start

print(f"Tempo totale 8-bit: {t_8:.2f}s")
print(f"Tempo totale 4-bit: {t_4:.2f}s")
```

📘 *Domande guida:*
- Quale modello risponde più rapidamente?
- La differenza di latenza è percepibile per un utente umano?
- In che misura la riduzione di precisione migliora la **usabilità pratica**?


In [9]:

import ollama
import time
import psutil
import subprocess
import re
import pandas as pd
import json
import os

model8 = "llama3.2:3b-instruct-q8_0"
model4 = "llama3.2:3b-instruct-q4_0"
model2 = "llama3.2:3b-instruct-q2_K"

def chat(model, prompt, temperature=0):
    start = time.perf_counter()

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "user", "content": prompt} #sono praticamente le istruzioni che diamo al modello, in questo caso gli diciamo di comportarsi come un utente e di rispondere al prompt che gli diamo
        ],
        options={
            "temperature": temperature # piu è vicina allo 0 piu sarà coerente e precisa la risposta, piu è vicina a 1 piu sarà creativa e fantasiosa la risposta
        }
    )

    elapsed = time.perf_counter() - start

    return response["message"]["content"].strip(), elapsed

In [10]:
''' 🧩 Esercizio 2 – Misurare la velocità di inferenza

**Obiettivo:** comprendere come la quantizzazione influisce sulla **velocità di risposta** del modello.

**Istruzioni:**
1. Usa lo **stesso prompt breve**, ad esempio:
   > "Chi ha scritto *I Promessi Sposi*?"
2. Misura il **tempo totale di generazione della risposta** per ciascuna variante: **modello 8-bit** e **modello 4-bit**.
3. Ripeti la misura **3 volte** per ogni modello e calcola la **media**.'''
import time
import ollama

# 1. Nomi dei modelli scaricati su Ollama
MODEL_8BIT = "llama3.2:3b-instruct-q8_0"  # Modifica con il nome esatto del tuo modello 8-bit
MODEL_4BIT = "llama3.2:3b-instruct-q4_0" # Modifica con il nome esatto del tuo modello 4-bit

# 2. Definizione delle funzioni per rendere il tuo codice eseguibile
def model_8bit(prompt_text):
    response = ollama.generate(model=MODEL_8BIT, prompt=prompt_text)
    return response['response']

def model_4bit(prompt_text):
    response = ollama.generate(model=MODEL_4BIT, prompt=prompt_text)
    return response['response']

# --- ESERCIZIO 2 ---
prompt = "Chi ha scritto I Promessi Sposi?"

# Misura per il modello 8-bit
t8 = []
for _ in range(3):
    start = time.time()
    res_8bit = model_8bit(prompt)  # Salva la risposta invece di scartarla con _
    _ = model_8bit(prompt)
    t8.append(time.time() - start)
print(f"Tempo medio 8-bit: {sum(t8)/3:.2f} s") #fa la somma del tempo e la divide per 3 dato che sono stati eseguiti 3 test
print(f"Risposta 8-bit:\n{res_8bit}\n")

# Misura per il modello 4-bit
t4 = []
for _ in range(3):
    start = time.time()
    res_4bit = model_4bit(prompt)  # Salva la risposta invece di scartarla con _
    _ = model_4bit(prompt)
    t4.append(time.time() - start)
print(f"Tempo medio 4-bit: {sum(t4)/3:.2f} s")
print(f"Risposta 4-bit:\n{res_4bit}\n")

Tempo medio 8-bit: 14.59 s
Risposta 8-bit:
I Promessi Sposi sono un romanzo storico scritto dall'autore italiano Alessandro Manzoni, pubblicato per la prima volta nel 1827. Il romanzo è ambientato nel XVII secolo in Lombardia e narra la storia di due giovani, Renzo Tramaglino e Lucia Mondella, che si innamorano e cercano di sposarsi nonostante le difficoltà e le ostacoli che li impediscono di realizzare il loro sogno. Il romanzo è considerato uno dei capolavori della letteratura italiana e ha avuto un grande impatto sulla cultura e sulla storia del paese.

Tempo medio 4-bit: 7.54 s
Risposta 4-bit:
"I Promessi Sposi" è un romanzo storico scritto da Alessandro Manzoni, pubblicato per la prima volta nel 1827. L'opera è ambientata nel XVII secolo, durante il periodo della peste a Milano, e narra la storia di due giovani innamorati, Renzo e Lucia, e il loro impegno per la giustizia e la libertà.

Manzoni, nato a Monza nel 1833, era un poeta e scrittore italiano che si è ispirato alle opere 

In [12]:
'''🧩 Esercizio 3 – Valutazione qualitativa della risposta (accuratezza)

**Obiettivo:** osservare l’effetto della quantizzazione sulla **qualità linguistica** delle risposte.

**Istruzioni:**
Prepara **5 prompt**, uno per ciascun tipo:
1. Domanda fattuale breve  
2. Spiegazione di un concetto semplice  
3. Riassunto di un testo di 6–8 frasi  
4. Produzione creativa (es. inizio di un racconto)  
5. Ragionamento leggero (es. “Se Marco ha 3 mele e ne dà una a Luca…”)

Esegui i prompt su **modello 8-bit** e **modello 4-bit**, confrontando:
- coerenza del testo,  
- completezza delle risposte,  
- correttezza delle informazioni.'''


prompts = [
    "Chi ha inventato il telefono?",
    "Spiega in modo semplice cos'è una rete neurale.",
    "Riassumi questo testo: [Il calcio è molto più di un semplice gioco: è una passione globale che unisce milioni di persone in tutto il mondo. Ogni fine settimana, gli stadi di tutto il pianeta prendono vita tra canti, bandiere ed emozioni intense. Sul terreno di gioco, ventidue giocatori si affrontano combinando strategia, tecnica e forza atletica. Il movimento del pallone detta il ritmo della sfida, trasformando ogni passaggio in un'occasione decisiva. Un singolo gol ha il potere di cambiare il destino di una partita e far esplodere la gioia dei tifosi. Oltre allo spettacolo, questo sport insegna il valore fondamentale del lavoro di squadra e del rispetto reciproco. Dai grandi stadi d'élite ai semplici campetti di periferia, basta un pallone per azzerare ogni differenza. È proprio questa sua magia imprevedibile a rendere il calcio un fenomeno universale e senza tempo.]",
    "Scrivi l'inizio di una storia ambientata su Marte.",
    "Se Marco ha 3 mele e ne dà una a Luca, quante ne rimangono?"
]

risposte = {"8bit": [], "4bit": []}
for p in prompts:
    risposte["8bit"].append(model_8bit(p))
    risposte["4bit"].append(model_4bit(p))
    

print("Risposte modello 8-bit:")
for i, r in enumerate(risposte["8bit"]):
    print(f"Prompt {i+1}: {r}\n")
print("Risposte modello 4-bit:")
for i, r in enumerate(risposte["4bit"]):
    print(f"Prompt {i+1}: {r}\n")


Risposte modello 8-bit:
Prompt 1: Ciao! Sono felice di aiutarti a scoprire chi ha inventato il telefono.

Il telefono è stato inventato da un uomo canadese di nome Alexander Graham Bell, nato il 3 marzo 1847 a Edinburgh, in Scozia. Bell era un ingegnere e un ricercatore, noto per il suo lavoro sull'elocinismo e sulla comunicazione.

Nel 1876, Bell lavorava per un istituto di ricercatori a Boston, in Massachusetts, dove stava conducendo esperimenti con l'elocinismo e la comunicazione. Durante questo periodo, stava cercando di trovare una soluzione per comunicare attraverso i segnali elettrici.

Bell ricevette il brevetto per il telefono il 7 marzo 1876, e il suo dispositivo iniziale utilizzava un microfono per catturare il suono e un interruttore per interrompere il segnale elettrico. Il suo telefono era in grado di trasmettere il suono da un luogo all'altro, aprendo la strada alla comunicazione a distanza.

Tuttavia, è importante notare che ci sono alcune controversie sulla paternità d

In [13]:
'''## 🧩 Esercizio 4 – Prompt di stress (robustezza con input lunghi)

**Obiettivo:** valutare come la **quantizzazione più spinta (4-bit)** reagisce a input complessi.

**Istruzioni:**
1. Prendi un testo di circa **200–300 parole** (es. un breve articolo o un estratto da Wikipedia).
2. Crea un prompt che chieda di **riassumere** o **estrarre i punti chiave**.
3. Esegui lo stesso prompt con **modello 8-bit** e **modello 4-bit**.
'''
long_prompt = "Riassumi il seguente testo in 5 punti chiave:\n\n[Sotto la superficie tranquilla di una foresta si nasconde un sistema di comunicazione e cooperazione sofisticato quanto la rete internet. Gli alberi, apparentemente creature solitarie e silenziose, sono in realtà fittamente interconnessi attraverso un intricata rete sotterranea formata da funghi micorrizici, nota tra gli studiosi come Wood Wide Web.Attraverso questo network biologico, le piante scambiano continuamente risorse vitali come acqua, carbonio, fosforo e azoto. Un imponente albero madre, situato al centro del bosco, è in grado di percepire se un giovane germoglio cresciuto all'ombra stia faticando a sopravvivere, inviandogli direttamente i nutrienti necessari per crescere. Tuttavia, il passaggio di risorse nutritive è solo una parte della storia: la rete funziona anche come un vero e proprio sistema di allarme precoce. Se una pianta viene attaccata da insetti o parassiti nocivi, invia istantaneamente segnali chimici ed elettrici lungo i filamenti fungini. Gli alberi vicini, ricevuta l'allerta, iniziano subito a sintetizzare sostanze chimiche di difesa per rendere le proprie foglie sgradevoli o tossiche prima ancora che l'attacco li raggiunga.Questa straordinaria scoperta ha rivoluzionato l'ecologia moderna. La classica visione biologica della natura come una spietata arena di sola competizione individuale lascia il posto a un modello fondato sulla mutua assistenza. Una foresta non è soltanto un insieme casuale di alberi, ma un superorganismo intelligente e resiliente, capace di proteggersi e prosperare attraverso la solidarietà delle sue radici.]"
print("Risposta 8-bit:")
print(model_8bit(long_prompt))
print("\nRisposta 4-bit:")
print(model_4bit(long_prompt))


Risposta 8-bit:
Ecco i 5 punti chiave del testo:

1. **La rete funzionale di comunicazione degli alberi**: gli alberi sono interconnessi attraverso un network biologico chiamato Wood Wide Web, formato da funghi micorrizici che consentono la scambiata di risorse vitali come acqua, carbonio, fosforo e azoto.

2. **L'albero madre: la guida dell'ecosistema**: l'imponente albero madre al centro del bosco è in grado di percepire e rispondere alle esigenze delle piante più giovani, inviando loro nutrienti e sostanze chimiche di difesa.

3. **Il sistema di allarme precoce**: quando una pianta viene attaccata da insetti o parassiti nocivi, invia istantaneamente segnali chimici ed elettrici lungo i filamenti fungini, allertando gli alberi vicini.

4. **La difesa collettiva**: gli alberi vicini ricevuta l'allerta iniziano a sintetizzare sostanze chimiche di difesa per rendere le proprie foglie sgradevoli o tossiche, proteggendo così se stessi e l'ecosistema.

5. **La scoperta rivoluzionaria: la s

In [14]:
'''## 🧩 Esercizio 5 – Misurare l’utilizzo di memoria in pratica

**Obiettivo:** verificare la riduzione del consumo RAM dovuta alla quantizzazione.

**Istruzioni:**
1. Avvia una sessione di inferenza con il **modello 8-bit** e poi con il **modello 4-bit**.
2. Durante l’esecuzione, osserva l’uso di memoria tramite strumenti di sistema (es. `top`, `htop` o Monitoraggio Attività su macOS).
3. Annota il valore massimo di RAM utilizzata e calcola la **differenza percentuale**.
'''

# Importa re per cercare le dimensioni nell'output di Ollama.
import re
# Importa subprocess per eseguire i comandi Ollama dal notebook.
import subprocess
# Importa time per attendere il caricamento e lo scaricamento dei modelli.
import time

# Usa un prompt uguale per entrambi i modelli, così il confronto è coerente.
PROMPT = "Spiega brevemente che cos'è la quantizzazione dei modelli linguistici."

# Legge l'output di `ollama ps` e cerca la riga relativa al modello richiesto.
def get_model_size_from_ps(model_name):
    # Esegue `ollama ps` e acquisisce l'output in formato testo.
    result = subprocess.run(
        ["ollama", "ps"],
        capture_output=True,
        text=True,
        check=False,
    )

    # Esamina le righe dell'output per trovare il nome del modello.
    for line in result.stdout.splitlines():
        if model_name in line:
            # Estrae la quantità di memoria e la relativa unità (GB o MB).
            match = re.search(r"(\d+(?:\.\d+)?)\s*(GB|MB)", line)
            if match:
                # Restituisce la dimensione mantenendo l'unità mostrata da Ollama.
                return f"{match.group(1)} {match.group(2)}"
            # Segnala che la riga del modello non contiene una dimensione riconoscibile.
            return "Dimensione non trovata"

    # Segnala che il modello non compare nell'output di `ollama ps`.
    return "Modello non trovato"

# Chiede a Ollama di scaricare dalla memoria il modello indicato.
def unload_model(model_name):
    subprocess.run(
        ["ollama", "stop", model_name],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )

# Converte le dimensioni in MB per poterle confrontare numericamente.
def size_to_mb(size):
    # Accetta valori nel formato restituito da Ollama, per esempio '2.8 GB'.
    match = re.fullmatch(r"(\d+(?:\.\d+)?)\s*(GB|MB)", size)
    if not match:
        return None

    # Converte il numero trovato in MB, se la dimensione era espressa in GB.
    value = float(match.group(1))
    return value * 1024 if match.group(2) == "GB" else value

# Conserva le dimensioni rilevate senza sovrascrivere la lista `results` usata prima.
memory_results = {}

# Ripete la stessa prova prima con l'8-bit e poi con il 4-bit.
for precision, model_name in (("8-bit", model8), ("4-bit", model4)):
    # Esegue il prompt tramite la funzione definita nella cella precedente.
    answer, elapsed, _ = run_ollama_prompt(model_name, PROMPT)
    print(f"\nRisposta {precision}: {answer}")

    # Attende che Ollama mantenga il modello caricato prima di leggere `ollama ps`.
    time.sleep(2)
    size = get_model_size_from_ps(model_name)
    print(f"Memoria occupata dal modello {precision}: {size}")

    # Salva la misura associandola alla precisione del modello.
    memory_results[precision] = size

    # Scarica il modello prima della prova successiva.
    unload_model(model_name)
    time.sleep(5)

# Converte entrambe le misure in MB prima di calcolare la percentuale.
size_8bit = size_to_mb(memory_results["8-bit"])
size_4bit = size_to_mb(memory_results["4-bit"])

# Calcola quanto la memoria del 4-bit è inferiore rispetto a quella dell'8-bit.
if size_8bit is not None and size_8bit > 0 and size_4bit is not None:
    reduction = (size_8bit - size_4bit) / size_8bit * 100
    print(f"\nRiduzione della memoria con il modello 4-bit: {reduction:.1f}%")
else:
    # Evita il calcolo se Ollama non ha restituito una delle due misure.
    print("\nDifferenza percentuale non calcolabile: manca una delle dimensioni.")


Risposta 8-bit: La quantizzazione dei modelli linguistici è un concetto fondamentale nella teoria linguistica che descrive il modo in cui i segni linguistici (fonemi, morfemi, sintagmi, ecc.) si quantificano e si combinano per formare le parole e le frasi.

Secondo questa teoria, i segni linguistici non sono trattati come entità continue, ma piuttosto come unità discrete che possono essere combinati in modo quantitativo per formare le parole e le frasi. Ciò significa che i linguisti utilizzano un sistema di numeri naturali (1, 2, 3, ecc.) per descrivere la quantità di ogni segno linguistico presente in una parola o una frase.

Ad esempio, se analizziamo la parola "cat", possiamo quantificare i segni linguistici come segue:

- /c/ (sonido "c")
- /a/ (sonido "a")
- /t/ (sonido "t")

In questo caso, la quantità di ogni segno linguistico è 1. Per la parola "cats", invece, i segni linguistici si quantificano come segue:

- /c/ (sonido "c") - 1
- /a/ (sonido "a") - 1
- /t/ (sonido "t") - 1


In [16]:
'''## 🧩 Esercizio 6 – Stabilità su sessioni prolungate

**Obiettivo:** valutare la **coerenza nel tempo** dei modelli quantizzati.

**Istruzioni:**
1. Scegli un argomento (es. “Rivoluzione industriale” o “Evoluzione delle stelle”).
2. Simula una **conversazione di 10–15 turni** con entrambi i modelli.
3. Analizza se il modello 4-bit perde coerenza o produce risposte più vaghe col passare dei turni.

💡 *Suggerimento tecnico:* puoi simulare una conversazione concatenando le risposte:'''


DOMANDE = [
    "Parliamo dell'evoluzione delle stelle. Dammi una panoramica generale.",
    "Come nasce una stella?",
    "Cos'è una nebulosa?",
    "Qual è il ruolo della gravità?",
    "Che differenza c'è tra una protostella e una stella adulta?",
    "Come funziona la fusione nucleare?"
    "Come si forma un buco nero stellare?",
    "Riassumi tutto quello che abbiamo discusso finora.",
    "Puoi costruire una timeline delle fasi evolutive?",
]


def run_conversation(model_name):

    messages = []

    transcript = []

    for turn, question in enumerate(DOMANDE, start=1):

        messages.append(
            {
                "role": "user",
                "content": question
            }
        )

        response = ollama.chat(
            model=model_name,
            messages=messages
        )

        answer = response["message"]["content"]

        messages.append(
            {
                "role": "assistant",
                "content": answer
            }
        )

        transcript.append({
            "turn": turn,
            "question": question,
            "answer": answer
        })

        print(f"\n=== TURNO {turn} ===")
        print(f"DOMANDA: {question}")
        print(f"RISPOSTA:\n{answer}")

    return transcript
# Esegue la stessa conversazione con entrambi i modelli e conserva le risposte.
conversazioni = {}
for nome_modello in (model8, model4):
    print(f"\n===== CONVERSAZIONE CON {nome_modello} =====")
    conversazioni[nome_modello] = run_conversation(nome_modello)



===== CONVERSAZIONE CON llama3.2:3b-instruct-q8_0 =====

=== TURNO 1 ===
DOMANDA: Parliamo dell'evoluzione delle stelle. Dammi una panoramica generale.
RISPOSTA:
Certo, sarò felice di darti una panoramica generale sull'evoluzione delle stelle.

L'evoluzione delle stelle è un processo che inizia con la nascita di una nuvola di gas e polvere nel buio spazio, e termina con la morte di una stella come una gigante rossa o come una supernova. Ecco una panoramica generale dei diversi stadi dell'evoluzione delle stelle:

**Stadio 1: Nascita (Protostella)**

* La nascita di una stella inizia con la formazione di una nube di gas e polvere.
* La nube si collassa sotto la sua stessa gravità, formando un protostar.
* Il protostar si accende come una stella a energia nucleare, emettendo radiazione e calore.

**Stadio 2: Stella principale (Main Sequence)**

* La stella principale è il stadio di vita più lungo di una stella.
* Durante questo stadio, la stella emette energia attraverso la fusione nucl

NameError: name 'ìù' is not defined

In [ ]:
*